# Bauteil-Validierung

Schneller Überblick über ein verarbeitetes Bauteil. Lädt ein bereits
durch die Pipeline gelaufenes `WeldVolumeModel` und visualisiert die
einzelnen Pipeline-Stufen zur Diagnose und Validierung.

**Workflow:** Pipeline aufrufen (z.B. `scripts/run_batch_subtraction.py`),
dann dieses Notebook auf das gewünschte Bauteil zeigen lassen.


## Konfiguration

In [ ]:
from pathlib import Path
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

from schweiss_ki.core.data_structures import WeldVolumeModel
from schweiss_ki.subtraction.plots import (
    plot_gap_profile,
    plot_segmentation_overview,
    plot_cross_section,
    plot_registration_overlay,
)

# ── Was geladen werden soll ────────────────────────────────────────────
MODEL_DIR = Path("../data/outputs/SCHWEIßSPALT_1,5_Spalt_I")
CAD_CACHE_DIR = Path("../data/outputs/Baugruppe_Beispielteil_V-Naht_1.5mm_Spalt")

# Optional: bekannter Soll-Wert für den Spaltprofil-Plot
GAP_SOLL_MM = 1.5

## 1. Modell laden und Übersicht

In [ ]:
model = WeldVolumeModel.load(MODEL_DIR)
print(model)
print()
print(f"Source-File:   {model.source_file}")
print(f"Source-Type:   {model.source_type}")
print(f"Punkte:        {model.n_points:,}")
print(f"Dichte:        {model.density:.3f} Punkte/mm³" if model.density else "Dichte:        n/a")
print(f"Hat Normalen:  {model.has_normals}")
print(f"Hat Labels:    {model.has_segmentation}")
print(f"Hat Subtract.: {model.has_subtraction}")
if model.has_subtraction:
    sub = model.subtraction_report
    print(f"CAD-Quelle:    {sub.cad_source_file}")

## 2. Preprocessing

Schritte und Punktreduktion aus dem `PreprocessingReport`.

In [ ]:
if not model.has_preprocessing:
    print("Kein Preprocessing-Report im Modell.")
else:
    pp = model.preprocessing_report
    print(f"Punkte:     {pp.points_in:,} → {pp.points_out:,}  "
          f"({pp.total_retention_rate:.1%} behalten)")
    print(f"Laufzeit:   {pp.total_duration_ms:.0f} ms")
    print()
    print(f"{'Step':<35} {'Punkte vor':>14} {'Punkte nach':>14} {'Retention':>12} {'Zeit':>10}")
    print("-" * 95)
    for step in pp.steps:
        print(f"{step.step_name:<35} {step.points_before:>14,} {step.points_after:>14,} "
              f"{step.retention_rate:>11.1%} {step.duration_ms:>8.1f} ms")

## 3. Segmentierung

Labels: 0 = Background, 1 = Flanke A, 2 = Flanke B, 3 = Spalt, 4 = Sub-Gap

In [ ]:
if not model.has_segmentation:
    print("Keine Labels im Modell.")
else:
    unique, counts = np.unique(model.labels, return_counts=True)
    print("Label-Verteilung:")
    for u, c in zip(unique, counts):
        name = (model.label_names or {}).get(int(u), f"label_{u}")
        print(f"  {int(u)}: {name:<15} {c:>10,}  ({c/len(model.labels)*100:5.1f} %)")

In [ ]:
fig = plot_segmentation_overview(model, view="top")
plt.show()

In [ ]:
# Optional: Seitenansicht für Z-Verteilung der Labels
fig = plot_segmentation_overview(model, view="side")
plt.show()

## 4. Registrierung

Scan im CAD-Koordinatensystem überlagert. Bei guter Registrierung
sollten Scan- (rot) und CAD-Punkte (blau) deckungsgleich liegen.

In [ ]:
if not model.has_subtraction:
    print("Kein Subtraction-Report – Registrierung nicht ausgeführt.")
else:
    sub = model.subtraction_report
    # Falls aus geladenem Modell (subtraction_report_raw): Stats aus den Roh-Daten
    raw = model.metadata.get("subtraction_report_raw", {})
    reg = raw.get("registration", {})
    print(f"Registrierungs-Steps:  {len(reg.get('steps', []))}")
    print(f"Final residual:        {reg.get('final_residual', 'n/a')} mm" if reg.get('final_residual') else "Final residual:        n/a")
    print(f"Gesamt-Laufzeit:       {reg.get('total_duration_ms', 'n/a')} ms" if reg.get('total_duration_ms') else "")
    print()
    for s in reg.get("steps", []):
        res = f"{s.get('residual', 'n/a'):.3f}mm" if s.get('residual') else "n/a"
        fit = f", fitness={s.get('fitness'):.3f}" if s.get('fitness') is not None else ""
        print(f"  - {s.get('step_name')}: {s.get('duration_ms', 0):.1f} ms, residual={res}{fit}")

In [ ]:
# CAD aus Cache laden für Overlay-Plot
cad_pcd = o3d.io.read_point_cloud(str(CAD_CACHE_DIR / "pointcloud.ply"))
print(f"CAD: {len(cad_pcd.points):,} Punkte, Normalen: {cad_pcd.has_normals()}")

fig = plot_registration_overlay(model, cad_pcd, view="top")
plt.show()

## 5. Spaltprofil

Wurzelspaltbreite entlang der Naht-Längsrichtung. Wenn `GAP_SOLL_MM`
gesetzt ist (siehe Konfigurations-Zelle), wird Soll-Linie + Toleranzband
mit eingezeichnet.

In [ ]:
fig = plot_gap_profile(model, gap_soll_mm=GAP_SOLL_MM)
if fig:
    plt.show()
else:
    print("Kein Spaltprofil im Modell verfügbar.")

## 6. Diagnose – Querschnitt

Y-Z-Querschnitt für einen X-Bereich entlang der Naht. Zeigt die
Punktverteilung der Labels und die linearen Extrapolationen, mit
denen die Spaltbreite berechnet wird.

Auffälligkeiten in dieser Ansicht:
- **Gratbildung**: Punkte am unteren Ende der Flanken knicken nach innen
- **Wurzeldurchhang**: Punkte unterhalb z=0 (Label 4 = Sub-Gap)
- **Asymmetrie**: Flanken sind nicht spiegelsymmetrisch
- **Wenig Punkte tief unten**: Wurzel war für den CMM-Strahl unzugänglich

In [ ]:
# X-Bereich variieren, um verschiedene Stellen entlang der Naht anzuschauen
X_MIN = -110.0
X_MAX = -100.0

fig = plot_cross_section(
    model,
    x_min=X_MIN, x_max=X_MAX,
    show_extrapolation=True,
)
plt.show()

### Mehrere Querschnitte vergleichen

Praktisch um zu sehen, wie sich das Profil entlang der Naht ändert.

In [ ]:
x_starts = [-180, -130, -80, -30]   # vier Querschnitte
width = 10.0   # mm

fig, axes = plt.subplots(1, len(x_starts), figsize=(20, 5), dpi=150)
for ax, x_start in zip(axes, x_starts):
    sub_fig = plot_cross_section(
        model, x_min=x_start, x_max=x_start + width,
        show_extrapolation=True,
    )
    # Inhalt vom sub_fig auf ax kopieren ist umständlich – pragmatisch:
    # die Funktion erzeugt eine eigene Figure. Hier zeigen wir sie einzeln.
    plt.close(sub_fig)

# Da plot_cross_section eigene Figuren erzeugt, einzeln zeigen:
for x_start in x_starts:
    fig = plot_cross_section(
        model, x_min=x_start, x_max=x_start + width,
        show_extrapolation=True,
        figsize=(7, 5),
    )
    plt.show()